# Comparaison de deux méthodes d'anonymisation

Dans le notebook 02, les noms sont masqués avec Presidio et des règles écrites à la main : on ne masque une détection de spaCy que si elle suit une civilité, contient un prénom connu ou est suivie d'un âge.

Autre méthode à tester : ajouter à Presidio un reconnaisseur médical, pour que les médicaments et les maladies ne soient plus pris pour des noms de personnes. Le code du ticket Presidio #1491 a depuis été intégré à Presidio : c'est la classe `MedicalNERRecognizer`, présente dans la version 2.2.366 qu'on utilise. Elle s'appuie sur le modèle `blaze999/Medical-NER` (DeBERTa, entraîné sur de l'anglais).

On compare les deux méthodes sur le même échantillon que la section 3 du notebook 02 : 300 exemples français et 300 anglais.

In [1]:
import json
import random
from pathlib import Path

import pandas as pd
import torch
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_analyzer.predefined_recognizers import MedicalNERRecognizer

SEED = 42
print("GPU :", torch.cuda.is_available())


def load(folder, name):
    return [json.loads(line) for line in open(Path("../data") / folder / f"{name}.jsonl", encoding="utf-8")]


def texts_of(row):
    if "messages" in row:
        return [m["content"] for m in row["messages"]]
    return [row["prompt"][0]["content"], row["chosen"][0]["content"], row["rejected"][0]["content"]]


# même tirage que dans le notebook 02 : avant anonymisation (interim) et après (processed)
before = load("interim", "sft_train") + load("interim", "dpo_train")
after = {r["id"]: r for r in load("processed", "sft_train") + load("processed", "dpo_train")}
rng = random.Random(SEED)
sample = rng.sample([r for r in before if r["langue"] == "fr"], 300) + rng.sample([r for r in before if r["langue"] == "en"], 300)
print(len(sample), "exemples")

C:\Users\gregh\Desktop\OpenClassrooms\AI Engineer\P14\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU : True


600 exemples


## 1. La méthode avec le reconnaisseur médical

Presidio par défaut (spaCy pour les noms, ses détecteurs d'e-mails et de téléphones) plus le reconnaisseur médical. Quand une détection « personne » chevauche un terme médical, on la garde : c'est la gestion du conflit PERSON / MEDICATION. Les termes médicaux eux-mêmes ne sont pas masqués, ils sont utiles pour l'entraînement.

Le modèle est anglais, mais on l'ajoute aussi pour le français pour voir ce qu'il donne.

In [2]:
provider = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "fr", "model_name": "fr_core_news_md"},
               {"lang_code": "en", "model_name": "en_core_web_md"}],
})
analyzer = AnalyzerEngine(nlp_engine=provider.create_engine(), supported_languages=["fr", "en"])
for lang in ("fr", "en"):
    analyzer.registry.add_recognizer(MedicalNERRecognizer(supported_language=lang))

MEDICAL = MedicalNERRecognizer.ENTITIES
ENTITIES = ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"] + MEDICAL


def medical_method(text, lang):
    # renvoie toutes les détections, celles à masquer, et les termes médicaux trouvés
    results = analyzer.analyze(text=text, language=lang, entities=ENTITIES)
    medical = [r for r in results if r.entity_type in MEDICAL]
    to_mask = []
    for r in results:
        # le modèle renvoie aussi d'autres étiquettes (âge, description...) : on ne masque que les identifiants
        if r.entity_type not in ("PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"):
            continue
        overlap = any(m.start < r.end and r.start < m.end for m in medical)
        if r.entity_type == "PERSON" and overlap:
            continue
        to_mask.append(r)
    return results, to_mask, medical

C:\Users\gregh\Desktop\OpenClassrooms\AI Engineer\P14\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:  22%|██▎       | 45/200 [00:00<00:00, 442.91it/s]

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 1379.72it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:  20%|██        | 41/200 [00:00<00:00, 409.68it/s]

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 1245.07it/s]

Le test simple d'abord : un nom de patient et un médicament dans la même phrase, en anglais puis en français.

In [3]:
tests = [
    ("en", "Mrs. Johnson, age 58, has been diagnosed with HER-2 positive metastatic breast cancer after first-line "
           "treatment with Trastuzumab/pertuzumab plus taxane."),
    ("fr", "Mme Martin, 45 ans, ostéo-arthropathie hypertrophiante de Pierre Marie, traitée par Doliprane."),
]
for lang, text in tests:
    _, to_mask, medical = medical_method(text, lang)
    print(text)
    print("  masqué :", [(r.entity_type, text[r.start:r.end]) for r in to_mask])
    print("  médical :", [(r.entity_type, text[r.start:r.end]) for r in medical], "\n")

Mrs. Johnson, age 58, has been diagnosed with HER-2 positive metastatic breast cancer after first-line treatment with Trastuzumab/pertuzumab plus taxane.
  masqué : [('PERSON', 'Johnson')]
  médical : [('MEDICAL_MEDICATION', ' taxane'), ('MEDICAL_MEDICATION', ' Trastuzumab/pertuzumab plus'), ('MEDICAL_DISEASE_DISORDER', ' breast cancer'), ('MEDICAL_DISEASE_DISORDER', ' HER')] 

Mme Martin, 45 ans, ostéo-arthropathie hypertrophiante de Pierre Marie, traitée par Doliprane.
  masqué : [('PERSON', 'Mme Martin'), ('PERSON', 'Pierre Marie')]
  médical : [('MEDICAL_MEDICATION', ' Doliprane'), ('MEDICAL_HISTORY', '-ar'), ('MEDICAL_HISTORY', 'pathie hyper'), ('MEDICAL_DISEASE_DISORDER', 'thro'), ('MEDICAL_DISEASE_DISORDER', ' ostéo')] 



## 2. Les deux méthodes sur l'échantillon

Pour chaque détection « personne » de spaCy, on regarde la décision des deux méthodes :
- méthode médicale : masquée, sauf si elle chevauche un terme médical
- nos règles : on considère la détection masquée si elle n'apparaît plus dans le texte anonymisé du notebook 02

In [4]:
rows = []
n_medical = {"fr": 0, "en": 0}
for row in sample:
    for text, text_after in zip(texts_of(row), texts_of(after[row["id"]])):
        results, to_mask, medical = medical_method(text, row["langue"])
        n_medical[row["langue"]] += len(medical)
        masked = {(r.start, r.end) for r in to_mask}
        for r in (r for r in results if r.entity_type == "PERSON"):
            span = text[r.start:r.end]
            rows.append({"id": row["id"], "langue": row["langue"], "texte": span,
                         "contexte": (text[max(0, r.start - 50):r.start] + "[[" + span + "]]" + text[r.end:r.end + 50]).replace("\n", " "),
                         "methode_medicale": (r.start, r.end) in masked, "nos_regles": span not in text_after})

comp = pd.DataFrame(rows)
print(len(comp), "détections « personne »")
print("termes médicaux trouvés :", n_medical)
print(pd.crosstab(comp["methode_medicale"], comp["nos_regles"], rownames=["méthode médicale masque"], colnames=["nos règles masquent"]))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


1074 détections « personne »
termes médicaux trouvés : {'fr': 2231, 'en': 7203}
nos règles masquent      False  True 
méthode médicale masque              
False                      464      4
True                       570     36


Les deux méthodes ne sont pas d'accord sur les cases hors diagonale. On les relit à la main dans `data/controle/` (fichier non publié, il contient les textes d'origine) : pour chaque détection, est-ce un vrai nom de personne ? Pour le groupe le plus gros, on relit 60 lignes tirées au hasard.

In [5]:
CTRL = Path("../data/controle")
only_medical = comp[comp["methode_medicale"] & ~comp["nos_regles"]].sort_values(["id", "contexte"])
only_ours = comp[~comp["methode_medicale"] & comp["nos_regles"]].sort_values(["id", "contexte"])
groups = {"masque_seulement_methode_medicale": only_medical.sample(min(60, len(only_medical)), random_state=SEED),
          "masque_seulement_nos_regles": only_ours}
for name, df in groups.items():
    path = CTRL / f"comparaison_{name}.csv"
    if not path.exists():
        df[["langue", "texte", "contexte"]].assign(vrai_nom="").to_csv(path, index=False, encoding="utf-8")
    print(name, ":", len(df), "lignes à relire")

masque_seulement_methode_medicale : 60 lignes à relire
masque_seulement_nos_regles : 4 lignes à relire


In [6]:
def initials(span):
    return " ".join(w[0] + "." for w in span.split())


for name in groups:
    path = CTRL / f"comparaison_{name}_annote.csv"
    if not path.exists():
        print(path.name, ": pas encore annoté")
        continue
    ann = pd.read_csv(path, encoding="utf-8")
    print(f"{name} : {len(ann)} lignes, {int(ann['vrai_nom'].sum())} vrais noms de personnes")
    # les vrais noms ne sont montrés que par leurs initiales
    shown = [t if not v else initials(t) for t, v in zip(ann["texte"], ann["vrai_nom"])]
    print("   exemples :", shown, "\n")

masque_seulement_methode_medicale : 60 lignes, 1 vrais noms de personnes
   exemples : ['Konzo', 'Cushing\n- C', 'A. Cholecystographie orale', 'Hegar', 'Treg', 'A. Converging ABLB Laddergram', 'E. A', 'E. Est', 'Ht', 'D. I.T.T. supérieure', 'C. Transient', 'CRP<5mg', 'Cytoscape', 'Température', 'S. P.', "B. S'", 'E. Vin\n\nUne', 'Fibrinogène', 'B. Diminue', 'Médiastin', "B. N'", 'Rifampin', 'Vi', 'Homocysteine', 'B. A', 'E. Grenouillette', 'AT1R', 'Policy Framework', 'E. Lavement', 'E. Est', 'Kidney Function', 'S. Typhi', 'Michaelis', 'MetaCore', 'E.', 'hyperclarté', 'A. Plèvre', 'E.', 'E. Insuffisance', 'D. La', 'E.', 'Biofilms', 'C. High SISI', 'D. Liposolubles', 'C. Dilatation', 'Holter', 'Treg', 'Autoencoders', "D. N'", 'Monsieur', 'Gaucher', 'Monocytogenes', 'A. Eternuements', 'C. Absence', 'Hypoglycemia', 'Vi', 'B. Survient', 'C. Est', 'C. A', 'miRNA'] 

masque_seulement_nos_regles : 4 lignes, 4 vrais noms de personnes
   exemples : ['J.', 'J.', 'P. é. A.', 'P.'] 



## 3. Bilan

Sur les 1 074 détections « personne » de l'échantillon, les deux méthodes en gardent 464 et en masquent 36 ensemble.

La méthode médicale masque 570 détections que nos règles laissent. Sur les 60 relues, une seule est un nom de personne (les initiales d'un auteur cité). Les autres sont des lettres de QCM (« E. Est »), des maladies (Konzo, Gaucher), un médicament (Rifampin) ou des résultats d'analyse (Fibrinogène). Le reconnaisseur médical ne protège que les termes qu'il reconnaît, et il en rate beaucoup. En français, le modèle découpe les mots (« ostéo », « thro » dans le test simple).

Nos règles masquent 4 détections que la méthode médicale laisse en clair : les prénoms des membres d'une famille dans une question de génétique. Le modèle les a pris pour des termes médicaux, donc la règle de conflit les a gardés.

Sur le test simple, les deux méthodes font pareil : « Johnson » masqué, Trastuzumab gardé, et « Pierre Marie » masqué alors que c'est un nom de maladie.

On garde nos règles pour le jeu publié. Le reconnaisseur médical repère bien les médicaments en anglais, mais utilisé seul il masquerait une grande partie du contenu médical. Ajouté à nos règles, il ne changerait sur cet échantillon que les 4 décisions ci-dessus, et dans le mauvais sens.

Limites : 60 lignes relues sur 570, un seul échantillon de 600 exemples, et un modèle médical entraîné sur de l'anglais.